# 09 - Module 4: ML baseline for LLM-agent validation

Train a classical ML classifier (Logistic Regression + Random Forest)
on the **same** C1/C2/C3 feature representation that the LLM agents see
in notebook 08, and measure whether its decisions agree with the agents
(Cohen's kappa). High agreement means the agents are doing something
defensibly rational; low agreement means they capture patterns the ML
baseline does not.

**Prerequisites:** notebooks 02a, 02b, 03a, 04, 05, 06, 08.


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

from xai_blockchain_framework.config import PATHS, AGENT_DISPLAY_NAMES, set_seed
from xai_blockchain_framework.metrics import cohen_kappa_pair

set_seed(42)

RESULTS_PATH = str(PATHS.results_dir)
FIGURES_PATH = str(PATHS.figures_dir)
PROCESSED_PATH = str(PATHS.data_processed)
for p in [RESULTS_PATH, FIGURES_PATH]:
    os.makedirs(p, exist_ok=True)

TOP_K = 5
AGENT_NAMES = [AGENT_DISPLAY_NAMES[s] for s in ('opus', 'gemini', 'gpt')]
print("Module 4  ML Baseline vs LLM Agents")
print("=" * 50)


## 1. Load inputs


In [ ]:
ell_splits = np.load(os.path.join(PROCESSED_PATH, 'elliptic_splits.npz'))
eth_splits = np.load(os.path.join(PROCESSED_PATH, 'ethereum_splits.npz'))
llm_df = pd.read_csv(os.path.join(RESULTS_PATH, 'module4v3_llm_raw_responses.csv'))
print(f'  LLM rows: {len(llm_df)}')

m1 = pd.read_csv(os.path.join(RESULTS_PATH, 'module1_fidelity_results.csv'))
m1 = m1[m1['k'] == TOP_K]
m2 = pd.read_csv(os.path.join(RESULTS_PATH, 'module2_stability_results.csv'))
m3 = pd.read_csv(os.path.join(RESULTS_PATH, 'module3_bras_results.csv'))

attrs_lookup = {
    ('RF',  'Elliptic', 'SHAP'): np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_elliptic.npy')),
    ('LGB', 'Elliptic', 'SHAP'): np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_elliptic.npy')),
    ('RF',  'Ethereum', 'SHAP'): np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_ethereum.npy')),
    ('LGB', 'Ethereum', 'SHAP'): np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_ethereum.npy')),
    ('RF',  'Elliptic', 'LIME'): np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_elliptic.npy')),
    ('LGB', 'Elliptic', 'LIME'): np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_elliptic.npy')),
    ('RF',  'Ethereum', 'LIME'): np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_ethereum.npy')),
    ('LGB', 'Ethereum', 'LIME'): np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_ethereum.npy')),
}

eval_idx_ell = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_elliptic.npy'))
eval_idx_eth = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_ethereum.npy'))


## 2. Feature construction helpers

Mirror the information exposed to the LLM agents in notebook 08:

- **C1** top-10 features by absolute magnitude only.
- **C2** C1 plus the model probability plus the top-5 XAI attributions.
- **C3** C2 plus the nine explanation-quality scores.


In [ ]:
QUALITY_KEYS = [
    'Comprehensiveness', 'Sufficiency', 'Infidelity',
    'Lipschitz_norm', 'Kendall_tau', 'Identity',
    'RAS', 'DVR', 'BRAS',
]


def get_quality_scores(model_name, dataset, xai):
    scores: dict[str, float] = {}
    for df in (m1, m2, m3):
        if df.empty:
            continue
        mask = pd.Series(True, index=df.index)
        if 'Model' in df.columns:   mask &= df['Model']   == model_name
        if 'XAI' in df.columns:     mask &= df['XAI']     == xai
        if 'Dataset' in df.columns: mask &= df['Dataset'] == dataset
        sub = df[mask]
        if sub.empty:
            continue
        row = sub.iloc[0]
        for k in QUALITY_KEYS:
            if k in row.index and pd.notna(row[k]):
                scores[k] = float(row[k])
    return scores


def build_ml_features(X, tx_indices, attrs, p_fraud, condition, quality):
    '''Build (X_cond, row_mask) for a vector of transaction indices.'''
    feats, row_mask = [], []
    for i, tx_idx in enumerate(tx_indices):
        if tx_idx >= len(X):
            row_mask.append(False); continue
        x = X[tx_idx]
        top10 = np.argsort(-np.abs(x))[:10]
        vec = x[top10].tolist()
        if condition in ('C2', 'C3'):
            vec.append(float(p_fraud[i]) if i < len(p_fraud) else 0.5)
            if attrs is not None and i < len(attrs):
                top5 = np.argsort(-np.abs(attrs[i]))[:TOP_K]
                vec.extend(attrs[i][top5].tolist())
            else:
                vec.extend([0.0] * TOP_K)
        if condition == 'C3':
            for k in QUALITY_KEYS:
                vec.append(float(quality.get(k, 0.0)))
        feats.append(vec)
        row_mask.append(True)
    return np.asarray(feats, dtype=np.float32), np.asarray(row_mask, dtype=bool)


## 3. Evaluate ML baselines against each LLM agent

Discover configurations dynamically from the Module 4 raw responses
(any combination of Dataset, BRAS_label, Model, XAI that actually
appears there).


In [ ]:
configs = (
    llm_df[['Dataset', 'Model', 'XAI', 'BRAS_label']]
    .drop_duplicates()
    .to_dict('records')
)
print(f'Found {len(configs)} configurations in llm responses')

all_rows: list[dict] = []
for cfg in configs:
    dataset, mn, xn, bl = cfg['Dataset'], cfg['Model'], cfg['XAI'], cfg['BRAS_label']
    splits = ell_splits if dataset == 'Elliptic' else eth_splits
    X_train, y_train = splits['X_train'], splits['y_train']
    X_test = splits['X_test']

    # Take the TX_idx + P_fraud from the C1/first-agent stream
    first_agent = AGENT_NAMES[0]
    sub = llm_df[
        (llm_df['Dataset'] == dataset)
        & (llm_df['BRAS_label'] == bl)
        & (llm_df['Model'] == mn) & (llm_df['XAI'] == xn)
        & (llm_df['Condition'] == 'C1') & (llm_df['Agent'] == first_agent)
    ]
    if sub.empty:
        continue
    tx_indices = sub['TX_idx'].to_numpy(dtype=int)
    p_fraud = sub['P_fraud'].to_numpy(dtype=float)
    y_test = splits['y_test']
    y_amb = y_test[tx_indices]

    attrs = attrs_lookup.get((mn, dataset, xn))
    quality = get_quality_scores(mn, dataset, xn)

    for condition in ('C1', 'C2', 'C3'):
        X_amb, mask_amb = build_ml_features(
            X_test, tx_indices, attrs, p_fraud, condition, quality,
        )
        if len(X_amb) == 0:
            continue
        # Training set uses all training rows; P(fraud) proxy = label so the
        # structure of the C2/C3 input matches.
        train_idx = np.arange(len(X_train))
        p_train = y_train.astype(float)
        X_tr, _ = build_ml_features(
            X_train, train_idx, attrs=None, p_fraud=p_train,
            condition=condition, quality=quality,
        )
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_amb_s = scaler.transform(X_amb)
        y_amb_masked = y_amb[mask_amb]
        tx_amb = tx_indices[mask_amb]

        for clf_name, clf in [
            ('LR', LogisticRegression(max_iter=1000, class_weight='balanced')),
            ('RF_baseline', RandomForestClassifier(
                n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)),
        ]:
            clf.fit(X_tr_s, y_train)
            y_pred = clf.predict(X_amb_s)
            da_ml = float(accuracy_score(y_amb_masked, y_pred))

            for agent in AGENT_NAMES:
                llm_sub = llm_df[
                    (llm_df['Dataset'] == dataset) & (llm_df['BRAS_label'] == bl)
                    & (llm_df['Model'] == mn) & (llm_df['XAI'] == xn)
                    & (llm_df['Condition'] == condition) & (llm_df['Agent'] == agent)
                ].set_index('TX_idx')
                if llm_sub.empty:
                    continue
                llm_dec, ml_dec, truth = [], [], []
                for j, tx in enumerate(tx_amb):
                    if tx not in llm_sub.index:
                        continue
                    d = llm_sub.loc[tx, 'Decision']
                    if isinstance(d, pd.Series):
                        d = d.iloc[0]
                    if pd.isna(d):
                        continue
                    llm_dec.append(1 if d == 'fraud' else 0)
                    ml_dec.append(int(y_pred[j]))
                    truth.append(int(y_amb_masked[j]))
                if len(llm_dec) < 2:
                    continue
                kappa = cohen_kappa_pair(
                    ['fraud' if v == 1 else 'legitimate' for v in ml_dec],
                    ['fraud' if v == 1 else 'legitimate' for v in llm_dec],
                )
                da_llm = float(accuracy_score(truth, llm_dec))
                all_rows.append({
                    'Dataset': dataset, 'Config': f'{mn}-{xn}',
                    'BRAS_label': bl, 'Condition': condition,
                    'ML_Classifier': clf_name, 'LLM_Agent': agent,
                    'DA_ML': da_ml, 'DA_LLM': da_llm,
                    'Kappa_ML_vs_LLM': float(kappa),
                    'N_compared': len(ml_dec),
                })
        print(f"  {dataset} {bl[:12]} {mn}-{xn} {condition}: {len(X_amb)} tx  feat_dim={X_amb.shape[1]}")

results_df = pd.DataFrame(all_rows)
print(f'\nTotal comparisons: {len(results_df)}')


## 4. Aggregate and interpret


In [ ]:
print('\n--- Per-condition summary ---')
print(results_df.groupby(['Condition', 'ML_Classifier']).agg({
    'DA_ML': 'mean', 'DA_LLM': 'mean',
    'Kappa_ML_vs_LLM': 'mean', 'N_compared': 'sum',
}).round(3).to_string())

print('\n--- Per-agent kappa ---')
print(results_df.groupby(['Condition', 'LLM_Agent']).agg({
    'Kappa_ML_vs_LLM': ['mean', 'std'],
    'DA_ML': 'mean', 'DA_LLM': 'mean',
}).round(3).to_string())

print('\n--- High vs low BRAS ---')
print(results_df.groupby(['BRAS_label', 'Condition']).agg({
    'Kappa_ML_vs_LLM': 'mean', 'DA_ML': 'mean', 'DA_LLM': 'mean',
}).round(3).to_string())

kappa_c2 = results_df[results_df['Condition'] == 'C2']['Kappa_ML_vs_LLM'].mean()
kappa_c3 = results_df[results_df['Condition'] == 'C3']['Kappa_ML_vs_LLM'].mean()
print(f'\nMean Kappa(ML<->LLM) C2: {kappa_c2:.3f}')
print(f'Mean Kappa(ML<->LLM) C3: {kappa_c3:.3f}')
if kappa_c2 > 0.6:
    print('  -> Substantial convergence: LLM validated as a reasonable decision-maker.')
elif kappa_c2 > 0.4:
    print('  -> Moderate convergence: LLM adds a complementary perspective.')
else:
    print('  -> Low convergence: the LLMs rely on patterns the ML baseline does not.')


## 5. Visualizations


In [ ]:
if not results_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for idx, cond in enumerate(['C1', 'C2', 'C3']):
        sub = results_df[results_df['Condition'] == cond]
        pivot = sub.pivot_table(values='Kappa_ML_vs_LLM',
                                index='ML_Classifier',
                                columns='LLM_Agent', aggfunc='mean')
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
                    vmin=-0.2, vmax=1.0, ax=axes[idx], cbar_kws={'shrink': 0.8})
        axes[idx].set_title(f'Kappa ML<->LLM  {cond}', fontweight='bold')
    plt.suptitle('ML baseline vs LLM agents agreement per condition',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, 'module4_ml_baseline_kappa.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('  wrote module4_ml_baseline_kappa.png')

    da_df = results_df.groupby('Condition').agg({'DA_ML': 'mean', 'DA_LLM': 'mean'}).reset_index()
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(da_df)); w = 0.35
    ax.bar(x - w / 2, da_df['DA_ML'], w, label='ML baseline', color='#3498db', alpha=0.85)
    ax.bar(x + w / 2, da_df['DA_LLM'], w, label='LLM mean', color='#e74c3c', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(da_df['Condition'])
    ax.set_ylabel('Decision Accuracy'); ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y'); ax.legend()
    ax.set_title('DA: ML baseline vs LLM agents', fontweight='bold')
    for bars in ax.containers:
        ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, 'module4_ml_baseline_da.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('  wrote module4_ml_baseline_da.png')


## 6. Save


In [ ]:
results_df.to_csv(os.path.join(RESULTS_PATH, 'module4_ml_baseline_results.csv'), index=False)
summary = (
    results_df.groupby(['Dataset', 'BRAS_label', 'Condition', 'ML_Classifier'])
    .agg({'DA_ML': 'first', 'DA_LLM': 'mean', 'Kappa_ML_vs_LLM': 'mean'})
    .round(3)
    .reset_index()
)
summary.to_csv(os.path.join(RESULTS_PATH, 'module4_ml_baseline_summary.csv'), index=False)
print('\nFiles saved:')
print('  module4_ml_baseline_results.csv (all comparisons)')
print('  module4_ml_baseline_summary.csv (aggregated)')
print('  module4_ml_baseline_kappa.png / module4_ml_baseline_da.png')
print('\nKappa guide:')
print('  >0.8  almost perfect    0.6-0.8 substantial   0.4-0.6 moderate   <0.4 low')
